In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/beraterolelk/thebest/__results__.html
/kaggle/input/notebooks/beraterolelk/thebest/__notebook__.ipynb
/kaggle/input/notebooks/beraterolelk/thebest/__output__.json
/kaggle/input/notebooks/beraterolelk/thebest/submission_god_tier_098120.csv
/kaggle/input/notebooks/beraterolelk/thebest/custom.css
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/__results__.html
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/submission.csv
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/__notebook__.ipynb
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/__output__.json
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/custom.css
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/catboost_info/test_error.tsv
/kaggle/input/notebooks/beraterolelk/0-975-ultimate-ensemble-lgbm-xgb-cb/catboost_info/learn_error.tsv
/kaggle/input/notebooks/beraterol

# OOF Meta-Stacking with Golden Features

This approach uses Out-of-Fold (OOF) predictions from three powerful gradient boosting frameworks (XGBoost, LightGBM, CatBoost) and feeds them into a Logistic Regression meta-learner to maximize the Balanced Accuracy metric.

### Part 1: Initialization & Data Loading
In this first cell, we import the necessary libraries, mute the warnings to keep our console clean, and load the dataset.

In [2]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Ignore warnings for a cleaner output
warnings.filterwarnings('ignore')

print("OOF Meta-Stacking Initiated...")

# --- 1. DATA LOADING ---
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
train = pd.read_csv(COMP + 'train.csv')
test = pd.read_csv(COMP + 'test.csv')
sub = pd.read_csv(COMP + 'sample_submission.csv')

OOF Meta-Stacking Initiated...


### Part 2: Feature Engineering (The Golden Features)
This step injects scientific domain knowledge into the dataset. Instead of letting the models guess the relationship between temperature and humidity, we mathematically calculate the Vapor Pressure Deficit (VPD), Water Deficit, and Plant Stress Index.

In [3]:
# --- 2. THE GOLDEN FEATURES ---
def engineer_features(df):
    out = df.copy()
    
    # Scientific / Domain Features
    # VPD (Vapor Pressure Deficit)
    out['VPD'] = np.exp((17.27 * out['Temperature_C']) / (out['Temperature_C'] + 237.3)) * (1 - (out['Humidity'] / 100))
    
    # Water Deficit & Stress Metrics
    out['Water_Deficit'] = out['Rainfall_mm'] - (out['Temperature_C'] * 0.5) 
    out['Total_Water'] = out['Rainfall_mm'] + out['Previous_Irrigation_mm']
    out['Stress_Index'] = out['Temperature_C'] / (out['Total_Water'] + 1e-5)
    
    # Cross-Combinations (Interactions)
    out['Crop_Soil_Combo'] = out['Crop_Type'].astype(str) + "_" + out['Soil_Type'].astype(str)
    
    # Categorical Encoding
    cats = [c for c in out.columns if out[c].dtype == object and c not in ['id', 'Irrigation_Need']]
    for c in cats:
        out[c] = out[c].astype('category').cat.codes
        
    return out

print("⚙️ Equipping the dataset with Golden Features...")
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Prepare Features (X) and Target (y)
X = train_fe.drop(columns=['id', 'Irrigation_Need'])
y = train_fe['Irrigation_Need'].map({"Low": 0, "Medium": 1, "High": 2}).values
X_test = test_fe.drop(columns=['id'])

⚙️ Equipping the dataset with Golden Features...


### Part 3: Level 1 Base Models (The Soldiers)
We initialize our three core gradient boosting models. Each of these models will learn different patterns from the engineered data. XGBoost and CatBoost utilize GPU acceleration, while LightGBM is set to utilize all CPU cores to prevent memory deadlocks in Kaggle kernels.

In [4]:
# --- 3. LEVEL 1: BASE MODELS (The Soldiers) ---
print("⚔️ Preparing Level 1 Base Models (XGB, LGBM, CB)...")

# GPU-supported heavy-weight parameters
xgb_model = XGBClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.03, 
    tree_method='hist', device='cuda', random_state=42
)

# Using CPU for LightGBM to prevent Kaggle GPU kernel deadlocks
lgb_model = LGBMClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.03, 
    n_jobs=-1, random_state=42, verbose=-1
)

cb_model = CatBoostClassifier(
    iterations=800, depth=6, learning_rate=0.03, 
    task_type='GPU', random_seed=42, verbose=False
)

level_1_models = [
    ('xgb', xgb_model),
    ('lgb', lgb_model),
    ('cb', cb_model)
]

⚔️ Preparing Level 1 Base Models (XGB, LGBM, CB)...


### Part 4: Level 2 Meta-Learner & The Stacking Engine
This is where the magic happens. The `LogisticRegression` acts as the Commander. It looks at the predictions of the three base models and learns who to trust and when. We use `class_weight='balanced'` to explicitly force the Meta-Learner to prioritize the minority "High" class.

In [5]:
# --- 4. LEVEL 2: META-LEARNER (The Commander) ---
# Logistic Regression will take the probabilities produced by the soldiers and make the flawless final decision.
# By using "class_weight='balanced'", we give special importance to the 'High' class!
meta_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# --- 5. THE STACKING ENGINE ---
print("🧠 Meta-Learner training is starting. (This might take a few minutes, sit back...)")

# 5-Fold Cross-Validation for flawless generalization
stacking_clf = StackingClassifier(
    estimators=level_1_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=1, # Keep at 1 to prevent GPU deadlocks
    passthrough=False # Use only the predictions from Level 1, not the original features
)

# Ignite the Training
stacking_clf.fit(X, y)

🧠 Meta-Learner training is starting. (This might take a few minutes, sit back...)


StackingClassifier(cv=5,
                   estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device='cuda',
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interactio...
                                              num_parallel_tree=None, ...)),
                               ('lgb',
                                LGBMClassifier(learning_rate=0.03, max_depth=6,
                                               n_estimators=800, n_jobs=-1,
                                               random_state=42, verbose=-1)),
                               ('cb',
                                CatBoostClassifier(depth=6, iterations=800, learning_rate=0.03, random_seed=42, task_type='GPU', verbose=False))],
                   final_estimator=LogisticRegression(class_weight='balanced',
                                                      max_iter=1000,
                                                      random_state=42),
                   n_jobs=1)

### Part 5: Final Predictions & Export
Once the stacking model is trained, we generate our final predictions for the test set, map the numeric values back to the original text labels, and save the submission file.

In [6]:
# --- 6. THE GRAND FINALE (Test Predictions) ---
print("\n🎯 Generating peak predictions...")
final_preds = stacking_clf.predict(X_test)

# Map numeric predictions back to string labels
idx2label = {0: "Low", 1: "Medium", 2: "High"}
sub['Irrigation_Need'] = np.vectorize(idx2label.get)(final_preds)

# Save the final submission file
sub.to_csv("submission_THE_ENDGAME_0982.csv", index=False)

print("\n🔥 ALL PROCESSES COMPLETED! 'submission_THE_ENDGAME_0982.csv' is ready.")
print("Prediction Distribution:\n", sub['Irrigation_Need'].value_counts())
print("\nThis code is the final bullet of a Kaggle Master. Submit the output file and walk to the podium! 🏆🥇")


🎯 Generating peak predictions...

🔥 ALL PROCESSES COMPLETED! 'submission_THE_ENDGAME_0982.csv' is ready.
Prediction Distribution:
 Irrigation_Need
Low       159635
Medium    101122
High        9243
Name: count, dtype: int64

This code is the final bullet of a Kaggle Master. Submit the output file and walk to the podium! 🏆🥇
